# Pixel-level bound tightness vs semantic regions



In [1]:
import os, re, glob
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ============================
# CONFIG
# ============================
device = "cpu"
eps = 0.2

DEBUG_DIR = "../../benchmarks/vggnet16_benchmark2022_segmented_one_img_k/debug_vis"
OUT_PDF = "bound_width_visualizations_2x2_heat_right_fixed_cbar.pdf"

# ============================
# PREPROCESS
# ============================
def center_crop_224(pil_img: Image.Image) -> Image.Image:
    """Resize to 256x256 then center crop to 224x224 (ImageNet-style)."""
    pil_img = pil_img.convert("RGB").resize((256, 256))
    left = (256 - 224) // 2
    top  = (256 - 224) // 2
    return pil_img.crop((left, top, left + 224, top + 224))

def to_tensor_01(pil_img_224: Image.Image) -> torch.Tensor:
    """Convert 224x224 PIL RGB to torch tensor in [0,1] with shape (1,3,224,224)."""
    arr = np.array(pil_img_224)
    x = torch.from_numpy(arr).float().to(device) / 255.0
    return x.permute(2, 0, 1).unsqueeze(0)

def compute_heat(x_01: torch.Tensor, eps: float) -> np.ndarray:
    """Compute (ub-lb) averaged over channels, given L_inf eps on all pixels."""
    mask = torch.ones((1, 1, 224, 224), dtype=torch.float32, device=device)
    lb = torch.clamp(x_01 - eps * mask, 0.0, 1.0)
    ub = torch.clamp(x_01 + eps * mask, 0.0, 1.0)
    heat = (ub - lb).mean(dim=1).squeeze(0).detach().cpu().numpy()
    return heat

# ============================
# FILE MATCHING
# ============================
orig_paths = sorted(glob.glob(os.path.join(DEBUG_DIR, "*_original.png")))
score_pat = re.compile(r"^(?P<base>.+)_seg0_score_(?P<score>\d+\.\d+)\.png$")

def best_seg_image(base: str):
    """Pick the seg0_score image with the highest score (if multiple)."""
    candidates = glob.glob(os.path.join(DEBUG_DIR, f"{base}_seg0_score_*.png"))
    best = None
    best_score = -1.0
    for p in candidates:
        m = score_pat.match(os.path.basename(p))
        if not m:
            continue
        s = float(m.group("score"))
        if s > best_score:
            best_score = s
            best = p
    return best, best_score

# ============================
# MAIN LOOP -> MULTI-PAGE PDF
# ============================
count = 0
skipped = []

with PdfPages(OUT_PDF) as pdf:
    for op in orig_paths:
        base = os.path.basename(op).replace("_original.png", "")

        seg_path, seg_score = best_seg_image(base)
        if seg_path is None:
            skipped.append((base, "missing seg0_score image"))
            continue

        mask_bw_path = os.path.join(DEBUG_DIR, f"{base}_seg0_mask_bw.png")

        # Load & crop to 224x224
        pil_orig = center_crop_224(Image.open(op))
        pil_seg  = center_crop_224(Image.open(seg_path))

        # Heatmap
        x_01 = to_tensor_01(pil_orig)
        heat = compute_heat(x_01, eps)

        # ============================
        # 2x2 PLOT (HEATMAP ON RIGHT, COLORBAR FIXED)
        # ============================
        fig, axes = plt.subplots(
            2, 2,
            figsize=(8.6, 8.2),
            gridspec_kw={"wspace": 0.06, "hspace": 0.18}
        )

        # ---- Top Left: Original
        axes[0, 0].imshow(pil_orig)
        axes[0, 0].set_title("Original", fontsize=10)
        axes[0, 0].axis("off")

        # ---- Top Right: Segmentation
        axes[0, 1].imshow(pil_seg)
        axes[0, 1].set_title(f"Overlay Mask (score={seg_score:.3f})", fontsize=10)
        axes[0, 1].axis("off")

        # ---- Bottom Left: BW Mask
        axes[1, 0].set_title("Black and White Mask", fontsize=10)
        if os.path.exists(mask_bw_path):
            pil_mask = center_crop_224(Image.open(mask_bw_path)).convert("L")
            axes[1, 0].imshow(pil_mask, cmap="gray", vmin=0, vmax=255)
        else:
            axes[1, 0].text(0.5, 0.5, "Mask not found", ha="center", va="center")
        axes[1, 0].axis("off")

        # ---- Bottom Right: Heatmap (RIGHT)
        axes[1, 1].imshow(pil_orig)
        im = axes[1, 1].imshow(heat, cmap="hot", alpha=0.6)
        axes[1, 1].set_title(f"Bound Width Heatmap (ε={eps})", fontsize=10)
        axes[1, 1].axis("off")

        # ---- Colorbar correctly aligned to the heatmap axis
        cbar = fig.colorbar(
            im,
            ax=axes[1, 1],
            fraction=0.046,  # thickness
            pad=0.04         # gap from the image
        )
        cbar.set_label("x_U - x_L", rotation=270, labelpad=12)

        # Title + layout
        # fig.suptitle(base, fontsize=11)
        # fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.95])

        # Save page
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
        count += 1

print(f"Saved {count} pages to {OUT_PDF}")
if skipped:
    print("Skipped items:")
    for b, why in skipped[:50]:
        print(" -", b, ":", why)
    if len(skipped) > 50:
        print(f" ... and {len(skipped)-50} more")

Saved 12 pages to bound_width_visualizations_2x2_heat_right_fixed_cbar.pdf


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --------------------------------------------------
# CHANGE THESE PATHS
# --------------------------------------------------

image_folder = "images/"
mask_folder  = "masks/"

n_samples = 6
random_seed = 42

# --------------------------------------------------

random.seed(random_seed)

# list files
image_files = sorted(os.listdir(image_folder))

# keep only image files
image_files = [f for f in image_files if f.endswith((".png",".jpg",".jpeg"))]

# randomly pick examples
sample_files = random.sample(image_files, min(n_samples, len(image_files)))

# --------------------------------------------------
# plot
# --------------------------------------------------

fig, axes = plt.subplots(len(sample_files), 2,
                         figsize=(7, 3*len(sample_files)))

if len(sample_files) == 1:
    axes = np.array([axes])

for i, fname in enumerate(sample_files):

    img_path = os.path.join(image_folder, fname)
    mask_path = os.path.join(mask_folder, fname)

    img = np.array(Image.open(img_path).convert("RGB"))
    mask = np.array(Image.open(mask_path).convert("L"))

    # Example score extraction
    score = "?"
    if "score" in fname:
        try:
            score = fname.split("score_")[1].split("_")[0]
        except:
            pass

    axes[i,0].imshow(img)
    axes[i,0].set_title("Original")
    axes[i,0].axis("off")

    axes[i,1].imshow(mask, cmap="gray")
    axes[i,1].set_title(f"Mask | score={score}")
    axes[i,1].axis("off")

plt.tight_layout()
plt.show()

Saved 12 pages to bw_mask_and_bound_width_EQUAL_IMAGES.pdf
